In [2]:
#Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Extract the dataset
import zipfile

zip_path = "/content/drive/MyDrive/thyroid.zip"
extract_path = "/content/thyroid"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [ ]:
import os
import xml.etree.ElementTree as ET

counts = {"0": 0, "1": 0}

xml_dir = "/content/thyroid/Annotations"

for xml_file in os.listdir(xml_dir):
    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(xml_dir, xml_file))
    root = tree.getroot()

    for obj in root.findall("object"):
        label = obj.find("name").text.strip()
        counts[label] += 1

print(counts)

{'0': 1426, '1': 3574}


In [ ]:
#Check folders
import os

for root, dirs, files in os.walk("/content/thyroid"):
    print(root)

/content/thyroid
/content/thyroid/ImageSets
/content/thyroid/ImageSets/Main
/content/thyroid/Annotations
/content/thyroid/JPEGImages


In [ ]:
#Count XML files
import os

annotation_files = os.listdir("/content/thyroid/Annotations")

print("Number of XML files:", len(annotation_files))
print(annotation_files[:5])

Number of XML files: 5000
['002329.xml', '004110.xml', '002658.xml', '001649.xml', '000779.xml']


In [ ]:
#View one XML
with open("/content/thyroid/Annotations/" + annotation_files[0], "r") as f:
    print(f.read()[:1000])

<annotation>
	<folder>VOC2007</folder>
	<filename>002329.jpg</filename>
	<source>
		<database>My Database</database>
		<annotation>VOC2007</annotation>
		<image>flickr</image>
		<flickrid>NULL</flickrid>
	</source>
	<owner>
		<flickrid>NULL</flickrid>
		<name>xiaoxianyu</name>
	</owner>
	<size>
		<width>718</width>
		<height>500</height>
		<depth>3</depth>
	</size>
	<segmented>0</segmented>
	<object>
		<name>1</name>
		<pose>Unspecified</pose>
		<truncated>0</truncated>
		<difficult>0</difficult>
		<bndbox>
			<xmin>222</xmin>
			<ymin>151</ymin>
			<xmax>380</xmax>
			<ymax>237</ymax>
		</bndbox>
	</object>
</annotation>



In [ ]:
#Read train/val/test split
with open('/content/thyroid/ImageSets/Main/train.txt') as f:
    train_ids = [x.strip() for x in f.readlines()]

with open('/content/thyroid/ImageSets/Main/val.txt') as f:
    val_ids = [x.strip() for x in f.readlines()]

with open('/content/thyroid/ImageSets/Main/test.txt') as f:
    test_ids = [x.strip() for x in f.readlines()]

print("Train:", len(train_ids))
print("Val:", len(val_ids))
print("Test:", len(test_ids))

Train: 3500
Val: 500
Test: 1000


In [ ]:
#Check dataset classes
import os
import xml.etree.ElementTree as ET

classes = set()

for xml_file in os.listdir("/content/thyroid/Annotations"):
    if xml_file.endswith(".xml"):

        tree = ET.parse(os.path.join("/content/thyroid/Annotations", xml_file))
        root = tree.getroot()

        for obj in root.findall("object"):
            classes.add(obj.find("name").text)

print(classes)

{'0', '1'}


In [ ]:
# step 8 #(Corrected): Convert XML to YOLO labels
import os
import xml.etree.ElementTree as ET

# Paths
xml_dir = "/content/thyroid/Annotations"
label_dir = "/content/thyroid/labels"

# Create labels directory
os.makedirs(label_dir, exist_ok=True)

# XML label -> YOLO class mapping
# Assuming:
# 0 = Benign
# 1 = Malignant
class_map = {
    "0": 0,
    "1": 1
}

for xml_file in os.listdir(xml_dir):

    if not xml_file.endswith(".xml"):
        continue

    tree = ET.parse(os.path.join(xml_dir, xml_file))
    root = tree.getroot()

    width = int(root.find("size/width").text)
    height = int(root.find("size/height").text)

    yolo_lines = []

    for obj in root.findall("object"):

        label = obj.find("name").text.strip()

        if label not in class_map:
            continue

        cls = class_map[label]

        xmin = float(obj.find("bndbox/xmin").text)
        ymin = float(obj.find("bndbox/ymin").text)
        xmax = float(obj.find("bndbox/xmax").text)
        ymax = float(obj.find("bndbox/ymax").text)

        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height
        w = (xmax - xmin) / width
        h = (ymax - ymin) / height

        yolo_lines.append(
            f"{cls} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"
        )

    txt_file = os.path.join(
        label_dir,
        xml_file.replace(".xml", ".txt")
    )

    with open(txt_file, "w") as f:
        f.write("\n".join(yolo_lines))

print("✅ XML converted to YOLO labels successfully!")

✅ XML converted to YOLO labels successfully!


In [ ]:
#step 9
import os

print("Label files:", len(os.listdir("/content/thyroid/labels")))

Label files: 5000


In [ ]:
#step 10
import os

base_dir = "/content/thyroid_yolo"

folders = [
    "images/train",
    "images/val",
    "images/test",
    "labels/train",
    "labels/val",
    "labels/test"
]

for folder in folders:
    os.makedirs(os.path.join(base_dir, folder), exist_ok=True)

print("Folders created!")

Folders created!


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'thyroid', 'thyroid_yolo', 'runs', 'drive', 'yolov8n.pt', 'sample_data']


In [ ]:
# step 11
with open('/content/thyroid/ImageSets/Main/train.txt') as f:
    train_ids = [x.strip() for x in f.readlines()]

with open('/content/thyroid/ImageSets/Main/val.txt') as f:
    val_ids = [x.strip() for x in f.readlines()]

with open('/content/thyroid/ImageSets/Main/test.txt') as f:
    test_ids = [x.strip() for x in f.readlines()]

In [ ]:
# step 12
import os
import shutil

images_dir = "/content/thyroid/JPEGImages"
labels_dir = "/content/thyroid/labels"

def copy_data(ids, split):

    for img_id in ids:

        img_src = os.path.join(images_dir, img_id + ".jpg")
        label_src = os.path.join(labels_dir, img_id + ".txt")

        img_dst = os.path.join(base_dir, "images", split, img_id + ".jpg")
        label_dst = os.path.join(base_dir, "labels", split, img_id + ".txt")

        if os.path.exists(img_src):
            shutil.copy(img_src, img_dst)

        if os.path.exists(label_src):
            shutil.copy(label_src, label_dst)

copy_data(train_ids, "train")
copy_data(val_ids, "val")
copy_data(test_ids, "test")

print("Dataset organized!")


Dataset organized!


In [ ]:
# step 13 (Corrected): Create data.yaml
data_yaml = """
path: /content/thyroid_yolo

train: images/train
val: images/val
test: images/test

nc: 2

names:
  0: benign
  1: malignant
"""

with open("/content/thyroid_yolo/data.yaml", "w") as f:
    f.write(data_yaml)

print("✅ data.yaml created successfully!")

✅ data.yaml created successfully!


In [ ]:
# step 14  Verify the dataset
import os

print("Train images :", len(os.listdir("/content/thyroid_yolo/images/train")))
print("Train labels :", len(os.listdir("/content/thyroid_yolo/labels/train")))

print("Validation images :", len(os.listdir("/content/thyroid_yolo/images/val")))
print("Validation labels :", len(os.listdir("/content/thyroid_yolo/labels/val")))

print("Test images :", len(os.listdir("/content/thyroid_yolo/images/test")))
print("Test labels :", len(os.listdir("/content/thyroid_yolo/labels/test")))

Train images : 3500
Train labels : 3500
Validation images : 500
Validation labels : 500
Test images : 1000
Test labels : 1000


In [ ]:
# step 15  Train YOLO
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/thyroid_yolo/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    workers=2,
    project="/content/runs",
    name="detect"
)

Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/thyroid_yolo/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=detect-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x790cf5e7ddc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
# step 16 Load the best model
from ultralytics import YOLO

model = YOLO("/content/runs/detect-2/weights/best.pt")

print("✅ Model loaded successfully!")

✅ Model loaded successfully!


In [ ]:
from shutil import copy

copy(
    "/content/runs/detect-2/weights/best.pt",
    "/content/drive/MyDrive/best.pt"
)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/best.pt")

print("✅ Model loaded from Google Drive!")

✅ Model loaded from Google Drive!


In [ ]:
import os

for root, dirs, files in os.walk("/content/runs"):
    if "best.pt" in files:
        print(os.path.join(root, "best.pt"))

/content/runs/detect-2/weights/best.pt


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/runs/detect-2/weights/best.pt")

results = model.predict(
    source="/content/thyroid_yolo/images/test",
    conf=0.25,
    save=True
)

for r in results[:10]:
    print("Image:", r.path)
    print("Predicted class:", r.boxes.cls.cpu().numpy())
    print("Confidence:", r.boxes.conf.cpu().numpy())
    print("-" * 30)


image 1/1000 /content/thyroid_yolo/images/test/000001.jpg: 448x640 1 benign, 40.8ms
image 2/1000 /content/thyroid_yolo/images/test/000006.jpg: 448x640 1 benign, 5.7ms
image 3/1000 /content/thyroid_yolo/images/test/000009.jpg: 448x640 1 malignant, 5.7ms
image 4/1000 /content/thyroid_yolo/images/test/000011.jpg: 512x640 1 benign, 38.7ms
image 5/1000 /content/thyroid_yolo/images/test/000016.jpg: 448x640 1 benign, 6.4ms
image 6/1000 /content/thyroid_yolo/images/test/000025.jpg: 576x640 1 benign, 1 malignant, 38.0ms
image 7/1000 /content/thyroid_yolo/images/test/000035.jpg: 448x640 1 benign, 6.3ms
image 8/1000 /content/thyroid_yolo/images/test/000040.jpg: 544x640 1 benign, 1 malignant, 39.5ms
image 9/1000 /content/thyroid_yolo/images/test/000044.jpg: 448x640 1 benign, 6.5ms
image 10/1000 /content/thyroid_yolo/images/test/000046.jpg: 448x640 1 malignant, 6.3ms
image 11/1000 /content/thyroid_yolo/images/test/000048.jpg: 448x640 1 malignant, 6.0ms
image 12/1000 /content/thyroid_yolo/images/te

In [ ]:
from ultralytics import YOLO
import os

model = YOLO("/content/runs/detect-2/weights/best.pt")

results = model.val()

print(results.results_dict)

Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1950.7±712.6 MB/s, size: 45.0 KB)
val: Scanning /content/thyroid_yolo/labels/val.cache... 500 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 500/500 174.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 3.8it/s 8.5s
                   all        500        500      0.802      0.841      0.849      0.556
                benign        125        125      0.743       0.76      0.772      0.482
             malignant        375        375      0.862      0.923      0.925       0.63
Speed: 2.8ms preprocess, 3.8ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /content/runs/detect/val
{'metrics/precision(B)': 0.80247890982289, 'metrics/recall(B)': 0.8413333333333333, 'metrics/mAP50(

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import os

# Load model
model = YOLO("/content/runs/detect-2/weights/best.pt")

image_dir = "/content/thyroid_yolo/images/test"
label_dir = "/content/thyroid_yolo/labels/test"

y_true = []
y_pred = []

images = sorted(os.listdir(image_dir))

for image_name in images:

    if not image_name.endswith(".jpg"):
        continue

    image_path = os.path.join(image_dir, image_name)

    label_path = os.path.join(
        label_dir,
        image_name.replace(".jpg", ".txt")
    )

    # Skip if no label
    if not os.path.exists(label_path):
        continue

    # Ground truth class (first object)
    with open(label_path) as f:
        line = f.readline().strip()

    if line == "":
        continue

    true_class = int(line.split()[0])

    # Prediction
    results = model.predict(
        image_path,
        conf=0.25,
        verbose=False
    )

    if len(results[0].boxes) == 0:
        continue

    # Highest confidence detection
    boxes = results[0].boxes

    confs = boxes.conf.cpu().numpy()
    classes = boxes.cls.cpu().numpy().astype(int)

    pred_class = classes[confs.argmax()]

    y_true.append(true_class)
    y_pred.append(pred_class)

print("Images evaluated:", len(y_true))

Images evaluated: 985


In [ ]:
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted"
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted"
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted"
)

cm = confusion_matrix(y_true, y_pred)

print("=" * 50)

print(f"Overall Accuracy : {accuracy*100:.2f}%")
print(f"Precision        : {precision*100:.2f}%")
print(f"Recall           : {recall*100:.2f}%")
print(f"F1 Score         : {f1*100:.2f}%")

print("=" * 50)

print("Confusion Matrix")
print(cm)

print("=" * 50)

print(classification_report(
    y_true,
    y_pred,
    target_names=["Benign", "Malignant"]
))

Overall Accuracy : 90.86%
Precision        : 90.71%
Recall           : 90.86%
F1 Score         : 90.73%
Confusion Matrix
[[206  56]
 [ 34 689]]
              precision    recall  f1-score   support

      Benign       0.86      0.79      0.82       262
   Malignant       0.92      0.95      0.94       723

    accuracy                           0.91       985
   macro avg       0.89      0.87      0.88       985
weighted avg       0.91      0.91      0.91       985

